<a href="https://colab.research.google.com/github/ashfaque0672/CNC_ML_Prediction_model/blob/main/CNC_ML_Model_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

VMC MACHINE PREDICTIVE MAINTENANCE - COMPLETE ML PIPELINE

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import joblib


STEP 1: CREATE SAMPLE MACHINE SENSOR DATA

In [ ]:
np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "Machine_ID": np.random.choice(
        ["M001", "M002", "M003", "M004"], n
    ),

    "Spindle_rpm": np.random.normal(3000, 300, n),

    "Spindle_bearing_temp": np.random.normal(65, 8, n),

    "Spindle_bearing_vib": np.random.normal(2.5, 0.7, n),

    "Spindle_motor_temp": np.random.normal(60, 7, n),

    "Spindle_motor_vib": np.random.normal(2.0, 0.5, n),

    "X_axis_vib": np.random.normal(1.8, 0.5, n),

    "Y_axis_vib": np.random.normal(1.7, 0.5, n),

    "Z_axis_vib": np.random.normal(1.6, 0.5, n),

    "Coolant_flow": np.random.normal(8, 1.5, n),

    "Pneumatic_pressure": np.random.normal(6, 0.7, n),

    "Power_consumption": np.random.normal(12, 2, n)
})


STEP 2: CREATE FAILURE LABEL

In [ ]:
# This is only for demonstration.
# In a real project, failure labels should come from actual
# maintenance/failure records.

failure_condition = (
    (df["Spindle_bearing_temp"] > 75) |
    (df["Spindle_bearing_vib"] > 3.5) |
    (df["Spindle_motor_vib"] > 3.0) |
    (df["Power_consumption"] > 16)
)

df["Failure"] = failure_condition.astype(int)


print(df.head())
print("\nFailure distribution:")
print(df["Failure"].value_counts())

  Machine_ID  Spindle_rpm  Spindle_bearing_temp  Spindle_bearing_vib  \
0       M003  3102.526793             75.413930             2.338720   
1       M004  3562.851252             77.492090             1.853037   
2       M001  3285.127151             65.256033             3.123139   
3       M003  2826.928903             58.972657             3.224675   
4       M003  2730.475599             68.679777             1.207668   

   Spindle_motor_temp  Spindle_motor_vib  X_axis_vib  Y_axis_vib  Z_axis_vib  \
0           54.286479           1.656860    2.475508    1.687801    1.441959   
1           60.547001           1.283664    1.043736    0.988885    1.757169   
2           66.031453           2.072918    1.134647    2.121257    1.819575   
3           60.973423           2.292650    0.711016    2.077479    2.016590   
4           48.961025           2.257442    1.450293    0.729071    2.241397   

   Coolant_flow  Pneumatic_pressure  Power_consumption  Failure  
0     10.445195     

STEP 3: FEATURE ENGINEERING

In [ ]:
# Temperature difference
df["Temperature_difference"] = (
    df["Spindle_bearing_temp"]
    - df["Spindle_motor_temp"]
)

# Total vibration
df["Total_vibration"] = (
    df["Spindle_bearing_vib"]
    + df["Spindle_motor_vib"]
    + df["X_axis_vib"]
    + df["Y_axis_vib"]
    + df["Z_axis_vib"]
)

# Average axis vibration
df["Average_axis_vibration"] = (
    df["X_axis_vib"]
    + df["Y_axis_vib"]
    + df["Z_axis_vib"]
) / 3

STEP 4: SELECT FEATURES

In [ ]:
features = [
    "Spindle_rpm",
    "Spindle_bearing_temp",
    "Spindle_bearing_vib",
    "Spindle_motor_temp",
    "Spindle_motor_vib",
    "X_axis_vib",
    "Y_axis_vib",
    "Z_axis_vib",
    "Coolant_flow",
    "Pneumatic_pressure",
    "Power_consumption",
    "Temperature_difference",
    "Total_vibration",
    "Average_axis_vibration"
]

X = df[features]
y = df["Failure"]

STEP 5: TRAIN / TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))



Training samples: 800
Testing samples: 200


STEP 6: HANDLE MISSING VALUES

In [ ]:
imputer = SimpleImputer(strategy="median")

STEP 7: CREATE ML MODEL

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

STEP 8: CREATE PIPELINE

In [ ]:
pipeline = Pipeline([
    ("imputer", imputer),
    ("model", model)
])

STEP 9: TRAIN MODEL

In [ ]:
pipeline.fit(X_train, y_train)

print("\nModel training completed.")


Model training completed.


STEP 10: PREDICTION

In [ ]:
y_pred = pipeline.predict(X_test)

y_probability = pipeline.predict_proba(X_test)[:, 1]

STEP 11: MODEL EVALUATION

In [ ]:
print("\n========== MODEL PERFORMANCE ==========")

print(
    "Accuracy:",
    round(accuracy_score(y_test, y_pred), 3)
)

print(
    "Precision:",
    round(precision_score(y_test, y_pred), 3)
)

print(
    "Recall:",
    round(recall_score(y_test, y_pred), 3)
)

print(
    "F1 Score:",
    round(f1_score(y_test, y_pred), 3)
)


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Normal", "Failure"]
    )
)


print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



========== MODEL PERFORMANCE ==========
Accuracy: 0.99
Precision: 1.0
Recall: 0.952
F1 Score: 0.976

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      1.00      0.99       158
     Failure       1.00      0.95      0.98        42

    accuracy                           0.99       200
   macro avg       0.99      0.98      0.98       200
weighted avg       0.99      0.99      0.99       200


Confusion Matrix:
[[158   0]
 [  2  40]]


STEP 12: FEATURE IMPORTANCE

In [ ]:
rf_model = pipeline.named_steps["model"]

importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n========== FEATURE IMPORTANCE ==========")
print(importance)


========== FEATURE IMPORTANCE ==========
                   Feature  Importance
1     Spindle_bearing_temp    0.302483
2      Spindle_bearing_vib    0.235905
11  Temperature_difference    0.109452
12         Total_vibration    0.068048
4        Spindle_motor_vib    0.066787
10       Power_consumption    0.051481
3       Spindle_motor_temp    0.030214
13  Average_axis_vibration    0.026957
5               X_axis_vib    0.019039
0              Spindle_rpm    0.018984
6               Y_axis_vib    0.018075
8             Coolant_flow    0.017844
9       Pneumatic_pressure    0.017750
7               Z_axis_vib    0.016982


STEP 13: SAVE TRAINED MODEL

In [ ]:
joblib.dump(
    pipeline,
    "vmc_predictive_maintenance_model.pkl"
)

print("\nModel saved successfully.")


Model saved successfully.


STEP 14: LOAD TRAINED MODEL

In [ ]:
loaded_model = joblib.load(
    "vmc_predictive_maintenance_model.pkl"
)

STEP 15: PREDICT NEW MACHINE DATA

In [ ]:
new_machine = pd.DataFrame({
    "Spindle_rpm": [3200],
    "Spindle_bearing_temp": [82],
    "Spindle_bearing_vib": [4.1],
    "Spindle_motor_temp": [70],
    "Spindle_motor_vib": [3.4],
    "X_axis_vib": [2.5],
    "Y_axis_vib": [2.4],
    "Z_axis_vib": [2.3],
    "Coolant_flow": [6.5],
    "Pneumatic_pressure": [5.5],
    "Power_consumption": [17]
})


# Apply exactly the same feature engineering

new_machine["Temperature_difference"] = (
    new_machine["Spindle_bearing_temp"]
    - new_machine["Spindle_motor_temp"]
)

new_machine["Total_vibration"] = (
    new_machine["Spindle_bearing_vib"]
    + new_machine["Spindle_motor_vib"]
    + new_machine["X_axis_vib"]
    + new_machine["Y_axis_vib"]
    + new_machine["Z_axis_vib"]
)

new_machine["Average_axis_vibration"] = (
    new_machine["X_axis_vib"]
    + new_machine["Y_axis_vib"]
    + new_machine["Z_axis_vib"]
) / 3


# Keep feature order exactly the same
new_machine = new_machine[features]


STEP 16: PREDICT

In [ ]:
prediction = loaded_model.predict(new_machine)[0]

probability = loaded_model.predict_proba(
    new_machine
)[0][1]


if prediction == 1:
    print("\n⚠️ PREDICTION: MACHINE FAILURE RISK")
else:
    print("\n✅ PREDICTION: MACHINE NORMAL")

print(
    "Failure probability:",
    round(probability * 100, 2),
    "%"
)


⚠️ PREDICTION: MACHINE FAILURE RISK
Failure probability: 93.5 %


                 VMC SENSOR DATA
                       │
                       ▼
                 AWS S3 / CSV
                       │
                       ▼
              Python / Pandas
                       │
                       ▼
              Data Preprocessing
              ├── Missing values
              ├── Outliers
              └── Data cleaning
                       │
                       ▼
              Feature Engineering
              ├── Total vibration
              ├── Temperature difference
              └── Average vibration
                       │
                       ▼
                Train / Test Split
                  80%       20%
                   │          │
                   ▼          │
             Random Forest    │
                   │          │
                   ▼          │
              Trained Model   │
                   │          │
                   └─────┬────┘
                         ▼
                    Evaluation
               Accuracy / Recall
               Precision / F1
                         │
                         ▼
                  Save Model (.pkl)
                         │
                         ▼
              New Sensor Reading
                         │
                         ▼
                 Failure Prediction